In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw09.ipynb")

# Homework 09: Text Analysis of Bloomberg Articles

## Data Cleaning and EDA


You must submit this assignment to Gradescope by the on-time deadline. **We strongly encourage you to submit your work to Gradescope several hours before the stated deadline.** This way, you will have ample time to reach out to staff for support if you encounter difficulties with submission. While course staff is happy to help guide you with submitting your assignment ahead of the deadline, we will not respond to last-minute requests for assistance.

Please read the instructions carefully when submitting your work to Gradescope.



## This Assignment

Welcome to Homework 09! For this assignment, we will work with Bloomberg news articles on Microsoft and Microsoft stock data (MSFT).

In this assignment, you will gain practice with:

- Conducting data cleaning and EDA on a text-based dataset,
- Manipulating data in `pandas` with the `datetime` and `string` accessors, and
- Writing regular expressions and using `pandas` RegEx methods.


In [ ]:
# Run this cell to set up your notebook. 
import warnings
warnings.simplefilter(action="ignore")

import re
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt



# Ensure that pandas shows at least 280 characters in columns, so we can see full articles.
pd.set_option("max_colwidth", 280)
plt.style.use("fivethirtyeight")
sns.set()
sns.set_context("talk")

## Before You Start

For each question in the assignment, please write down your answer in the answer cell(s) right below the question.

We understand that it is helpful to have extra cells breaking down the process towards reaching your final answer. If you happen to create new cells below your answer to run code, **NEVER** add cells between a question cell and the answer cell below it. It will cause errors when we run the autograder, and it will sometimes cause a failure to generate the PDF file.

**Important note: The local autograder tests will not be comprehensive. You can pass the automated tests in your notebook but still fail tests on Gradescope after the grades are released.** Please be sure to check your results carefully.

Finally, unless we state otherwise, **do not use for loops or list comprehensions**. The majority of this assignment can be done using built-in commands in `pandas` and `NumPy`.

### Debugging Guide

If you run into any technical issues, we highly recommend checking out the [Debugging Guide](https://mtu.instructure.com/courses/1571598/pages/debugging-guide). In this guide, you can find general questions about Jupyter notebooks / Datahub, Gradescope, and common pandas errors.

<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

## Question 1: Importing the Data

The data for this assignment is a subset of the financial news dataset from [this github repo](https://github.com/philipperemy/financial-news-dataset). The original datasets are no longer available online due to copyright issues, but we were allowed access for educational purposes. The data in the file `data/msft_bloomberg_news.txt` has been filtered to just Bloomberg articles published between 2010 to 2013 (inclusive) with text that contains "Microsoft" or "MSFT" (Microsoft's stock name).


<br>

---

### Question 1a

Let's examine the contents of the `data/msft_bloomberg_news.txt` file. Using the [`open` function](https://docs.python.org/3/library/functions.html#open) and [`read` operation](https://docs.python.org/3/tutorial/inputoutput.html#methods-of-file-objects) on a `python` file object, read **the first 1000 characters** in `data/msft_bloomberg_news.txt` and store your result in the variable `q1a`. Then, display the result so you can read it.

**CAUTION: Viewing the contents of large files in a Jupyter Notebook could crash your browser. Be careful not to print the entire contents of the file.**


In [ ]:
...
print(q1a)

In [ ]:
grader.check("q1a")

<br>

---

### JSON

The printed output you got from `q1a` is in the JSON format.

- JSON stands for **JavaScript Object Notation**. It is a lightweight format for storing and transferring data.

- It is:
    - very easy for computers to read and write.
    - moderately easy for programmers to read and write by hand.
    - meant to be generated and parsed.

- Most modern languages have an interface for working with JSON objects.
    - JSON objects _resemble_ Python dictionaries (but are not the same!).
 
#### JSON data types

| Type | Description |
| --- | --- |
| String | Anything inside double quotes. |
| Number | Any number (no difference between ints and floats). |
| Boolean | `true` and `false`. |
| Null | JSON's empty value, denoted by `null`. |
| Array | Like Python lists. |
| Object | A collection of key-value pairs, like dictionaries. Keys must be strings, values can be anything (even other objects). |

See [json-schema.org](https://json-schema.org/understanding-json-schema/reference/type.html) for more details.

**CAUTION: As a reminder, viewing the contents of large files in a Jupyter Notebook could crash your browser. Be careful not to print the entire contents of the file, and do not use the file explorer to open data files directly.**




<br>

---

### Question 1b

`pandas` has built-in readers for many different file formats, including the file format used here to store news articles. To learn more about these, check out the documentation for

- `pd.read_csv` [(docs)](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)
- `pd.read_html`[(docs)](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_html.html)
- `pd.read_json`[(docs)](https://pandas.pydata.org/docs/reference/api/pandas.read_json.html)
- `pd.read_excel`[(docs)](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_excel.html).

For this question, use one of these functions to:
1. Load the file `msft_bloomberg_news.txt` in the data folder as a `DataFrame` into the variable `msft_news_df`.
2. Set the **index** of `msft_news_df` to correspond to the `id` of each news article.


**Hint:** If your code is taking a while to run, you may have used the incorrect data loading function for the type of the given file.

In [ ]:
msft_news_df = ...
msft_news_df.head(1)

In [ ]:
grader.check("q1b")

<!-- BEGIN QUESTION -->



<br>

---

### Question 1c

Suppose we are interested in using the news to predict future stock values. What additional data would we need to predict stock prices, and how could we connect that data to news articles? In addition, what attributes or characteristics of the news might help predict the stock value? 

_Type your answer here, replacing this text._

<!-- END QUESTION -->



<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

## Question 2: Time Analysis

After loading in the data, we can start exploring news articles by analyzing the relationships between the release dates (date of publication) and different topics and companies.

<br>


<br>

---

### Question 2a

First, let's extract the date and time from the `released_at` column in `msft_news_df`. Notice that the date and time are encoded in the following format:

```
<date>May 29 2012</date> <time>09:40:58</time>
<date>May 18 2011</date> <time>22:42:40</time>
<date>August 15 2012</date> <time>00:09:02</time>
<date>July 1 2011</date> <time>22:12:37</time>
...
```

There are several ways to convert this to a `Timestamp` object that we can use more easily. However, for this assignment, we are going to use string manipulation functions. 

Create a regular expression that extracts the Month, Day, Year, Hour, Minute, and Second from the `msft_news_df["released_at"]` column. You should create a new `DataFrame` called `dates` that contains:
1. The same index as `msft_news_df` (`id`) and
2. Column labels: `"Month"`, `"Day"`, `"Year"`, `"Hour"`, `"Minute"`, `"Second"`.

Additionally, convert all numerical values (`"Year"`, `"Day"`, `"Hour"`, `"Minute"`, `"Second"`) to type `int`.

**Hint 1:** You should use the [`Series.str.extract`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.extract.html) function.

**Hint 2:** Don't forget to use raw strings and capture groups. Copy the above example text into [regex101.com](https://regex101.com/) to experiment with your regular expressions.

**Hint 3:** It might be helpful to break this up into a couple of steps (e.g., first extract date values such as Month, Day, and Year and then extract time values such as Hour, Minute, and Second).

In [ ]:
...

In [ ]:
grader.check("q2a")

<br>

---

### Question 2b

Now that we've figured out how to extract dates, create a new `DataFrame` called `msft_news_2010` that only contains articles released in 2010. This `DataFrame` should contain:
1. An index of `id` and
2. Columns: `"title"`, `"released_at"`, `"content"`, `"path"`, `"Month"`, `"Day"`, and `"Year"`.

**Hint:** Consider merging `msft_news_df` with `dates`.

In [ ]:
...

msft_news_2010.head(1)

In [ ]:
grader.check("q2b")

<br>

---

### Question 2c

After processing the article release dates, we can analyze articles about different topics and companies. Note that all the articles in the provided dataset mention Microsoft/MSFT, but they can also mention other companies.


For each company in the list of `companies` (provided below), add a boolean column to the `msft_news_df` `DataFrame` indicating whether the corresponding company is mentioned in the text of the article. Ultimately, you should add six new columns containing `True`/`False` values to the `DataFrame`: `"amazon"`, `"nintendo"`, `"apple"`, `"sony"`, `"facebook"`, `"netflix"`. You may use a for loop over the list of companies.

**Note:** Make the contents of the articles lowercase before searching for the keywords.

In [ ]:
companies = ["amazon", "nintendo", "apple", "sony", "facebook", "netflix"]

...

msft_news_df.head(5)

In [ ]:
grader.check("q2c")

<br>

---

### Question 2d

Now, we can put everything together to analyze the release dates and volume of articles for different companies.


Create a new `DataFrame` called `year_news` that contains the number of articles mentioning each company in the list `companies` after 2010 (inclusive). `year_news` should have six columns (one column for each company), and the index of this `DataFrame` should be the release year `"Year"`.

In [ ]:
year_news = ...
year_news.head()

In [ ]:
grader.check("q2d")

<!-- BEGIN QUESTION -->

### Question 2e 
#### Compute Company Share and Dominance per Year

Using the DataFrame **`year_news`** from the previous question, which contains yearly
counts of company mentions for `amazon`, `apple`, `facebook`, `netflix`, `nintendo`, and `sony`:

#### Tasks
1. Compute the **percentage share** of mentions for each company per year  
   (so that each row sums to 100%).
2. Add a new column `'TopCompany'` that identifies which company had the highest share each year.
3. Compute how many years each company was **dominant** — i.e., appeared as `'TopCompany'`.

#### Expected Variables
- `share_news` → DataFrame of normalized shares (in percentages)
- `dominance` → Series showing how many years each company dominated

#### Hints
- Use `.div(..., axis=0)` to normalize each row by its total sum.  
- Use `.idxmax(axis=1)` to find the top company in each row.  
- Use `.value_counts()` to count how many times each company appears as `'TopCompany'`.


In [ ]:
# 1. Compute percentage share
share_news = ...

# 2. Identify top company per year
share_news['TopCompany'] = ...

# 3. Compute dominance count
dominance = ...



In [ ]:
# do not edit/delete
share_news

In [ ]:
# do not edit/delete
dominance

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Question 2f 
#### Visualize Company Mention Trends (Line Plot)

Using the DataFrame **`year_news`** from the previous question, visualize how the
number of articles mentioning each company changed from **2010 to 2013**.

#### Tasks
1. Create a **line plot** showing the number of articles for each company across years.
2. Each company should have its own colored line with a legend.
3. Set:
   - `Year` on the x-axis
   - `Number of Articles` on the y-axis
   - A descriptive plot title  
   - Proper axis labels and ticks for each year (2010–2013)

#### Hints
- Use `sns.lineplot()` inside a loop to plot each company’s data.
- Use `plt.figure(figsize=(6,4))` to control figure size.
- Call `plt.legend()`, `plt.xlabel()`, `plt.ylabel()`, and `plt.title()` to make the plot clear.


In [ ]:
# TODO:
# 1. Create a line plot showing article counts per company across 2010–2013
# 2. Include legend, axis labels, title, and year ticks

plt.figure(figsize=(6, 4))
for company in companies:
    ...
    ...
    
plt.legend(fontsize="12")
plt.xticks(np.arange(2010, 2014), np.arange(2010, 2014))
plt.ylabel("Number of Articles")
plt.xlabel("Year")
plt.title("Number of Articles Released (2010–2013)")
plt.show()


<!-- END QUESTION -->

<!-- END QUESTION -->

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## You have finished Homework 09!



### Submission Instructions

Below, you will see a cell. Running this cell will automatically generate a zip file with your autograded answers. Submit this file to the HW 09  assignment on Gradescope. 

**You are responsible for ensuring your submission follows our requirements. We will not be granting regrade requests nor extensions to submissions that don't follow instructions.** If you encounter any difficulties with submission, please don't hesitate to reach out to staff prior to the deadline.


## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)